# RemoGen Studio - voice cloning on Colab (XTTS-v2)

Runs XTTS-v2 here on Colab's GPU for narration + voice cloning from your uploaded sample.
Your PC keeps doing everything else (script writing, preview, video rendering).

**How to use**
1. **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells below **in order**. The last cell prints a **URL** and an **API key**.
3. In RemoGen Studio open **Voice Engine**, paste both, press **Test connection**, and **Save**.
4. Leave this tab open while you generate. If the notebook stops, re-run the last cell - the URL and key **change every time**.

*Note:* XTTS-v2 is released under the Coqui Public Model License, which is **non-commercial**.

In [ ]:
# 1) Check for a GPU and install the dependencies (first run: ~2-4 minutes)
!nvidia-smi -L || echo ">>> NO GPU: choose Runtime -> Change runtime type -> T4 GPU, then re-run"
!pip -q install coqui-tts fastapi "uvicorn[standard]" python-multipart

In [ ]:
%%writefile rg_common.py
"""
Shared plumbing for RemoGen's Colab TTS servers (server_xtts.py, server_dia.py,
server_chattts.py). All three expose the SAME contract:
  GET  /health   -> {"status": "ok" | "loading", "model", "device", "gpu"}
  POST /voices   -> multipart "file" -> {"voice_id": "..."}
  POST /tts      -> JSON {"text", "language", "speed", "voice_id"} -> audio/wav
so the Node backend (services/tts/colabProvider.js) never needs to know or
care which model is actually running - only this file's contents differ.
"""
import glob
import hashlib
import hmac
import io
import math
import os
import struct
import subprocess
import wave
from typing import Optional

from fastapi import Header, HTTPException

API_KEY = os.environ.get("RG_API_KEY", "")
FAKE = os.environ.get("RG_FAKE_TTS") == "1"
VOICE_DIR = os.environ.get("RG_VOICE_DIR", "/tmp/rg_voices")
os.makedirs(VOICE_DIR, exist_ok=True)


def require_key(x_api_key: str = Header(default="")):
    if API_KEY and not hmac.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key")


def voice_path(voice_id: str) -> Optional[str]:
    """Resolves a voice_id to its uploaded file, refusing anything that isn't
    a plain hex id - blocks path-traversal tricks like "../../etc/passwd"."""
    if not voice_id or not voice_id.isalnum():
        return None
    matches = glob.glob(os.path.join(VOICE_DIR, f"{voice_id}.*"))
    return matches[0] if matches else None


def save_uploaded_voice(data: bytes, original_filename: str) -> str:
    """Saves an uploaded reference sample, deduped by content hash so
    re-uploading the same file is free, and normalises it to mono 22.05kHz
    (what these models expect) when ffmpeg is available."""
    if not data:
        raise HTTPException(status_code=400, detail="Empty file")
    if len(data) > 50 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Reference audio is over 50 MB")

    voice_id = hashlib.sha1(data).hexdigest()[:16]
    ext = os.path.splitext(original_filename or "")[1].lower() or ".wav"
    raw_path = os.path.join(VOICE_DIR, f"{voice_id}{ext}")
    with open(raw_path, "wb") as fh:
        fh.write(data)

    wav_path = os.path.join(VOICE_DIR, f"{voice_id}.wav")
    if raw_path != wav_path:
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path, "-ac", "1", "-ar", "22050", wav_path],
                check=True, timeout=120,
            )
            os.remove(raw_path)
        except Exception:
            pass  # keep the original file; most loaders can read it directly
    return voice_id


def wav_duration_seconds(path: str) -> float:
    with wave.open(path, "rb") as w:
        return w.getnframes() / w.getframerate()


def apply_speed(input_path: str, output_path: str, speed: float):
    """Re-times narration with ffmpeg's atempo filter. Used uniformly across
    all three engines rather than each model's own (inconsistent, sometimes
    absent) speed control, so behaviour matches what services/tts/http.js's
    caller expects regardless of which Colab engine is running."""
    speed = max(0.5, min(2.0, float(speed or 1.0)))
    if abs(speed - 1.0) < 0.01:
        if input_path != output_path:
            os.replace(input_path, output_path)
        return
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", input_path, "-filter:a", f"atempo={speed}", output_path],
        check=True, timeout=120,
    )


def fake_wav_bytes(text: str) -> bytes:
    """A short sine-wave tone standing in for real audio, so the HTTP
    plumbing (auth, upload, download) can be tested without a GPU or any of
    these (large) models actually installed. Set RG_FAKE_TTS=1 to use this."""
    seconds = max(0.6, 0.4 * len(text.split()))
    rate = 24000
    frames = b"".join(
        struct.pack("<h", int(6000 * math.sin(2 * math.pi * 220 * i / rate))) for i in range(int(seconds * rate))
    )
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(frames)
    return buf.getvalue()


In [ ]:
%%writefile server_xtts.py
"""
RemoGen Studio - XTTS-v2 voice-cloning server (runs in Google Colab).
See rg_common.py for the shared HTTP contract all three engine servers share.

XTTS-v2 does genuine zero-shot cloning from your uploaded reference sample
(no transcript needed) and has its own native speed control, so this is the
simplest of the three servers.
"""
import os
import threading

from fastapi import Depends, FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel
from typing import Optional
from contextlib import asynccontextmanager

from rg_common import API_KEY, FAKE, VOICE_DIR, require_key, voice_path, save_uploaded_voice, fake_wav_bytes

MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
state = {"tts": None, "device": "cpu", "gpu": None, "default_speaker": None, "error": None}
synth_lock = threading.Lock()  # one synthesis at a time; the GPU is the bottleneck anyway


def load_model():
    try:
        if FAKE:
            state.update(device="fake", gpu="none (fake mode)")
            state["tts"] = "fake"
            return
        os.environ.setdefault("COQUI_TOS_AGREED", "1")  # Coqui Public Model License (non-commercial)
        import torch
        from TTS.api import TTS

        device = "cuda" if torch.cuda.is_available() else "cpu"
        state["device"] = device
        state["gpu"] = torch.cuda.get_device_name(0) if device == "cuda" else None
        tts = TTS(MODEL_NAME).to(device)
        speakers = list(getattr(tts, "speakers", None) or [])
        state["default_speaker"] = os.environ.get("DEFAULT_SPEAKER") or (speakers[0] if speakers else None)
        state["tts"] = tts
    except Exception as exc:  # surfaced through /health so it isn't a silent hang
        state["error"] = f"{type(exc).__name__}: {exc}"
        raise


@asynccontextmanager
async def lifespan(_app):
    threading.Thread(target=load_model, daemon=True).start()
    yield


app = FastAPI(title="RemoGen Colab TTS - XTTS-v2", lifespan=lifespan)


@app.get("/health", dependencies=[Depends(require_key)])
def health():
    if state["error"]:
        return JSONResponse(status_code=500, content={"status": "error", "detail": state["error"]})
    return {
        "status": "ok" if state["tts"] is not None else "loading",
        "model": "xtts_v2" if not FAKE else "fake",
        "device": state["device"],
        "gpu": state["gpu"],
    }


@app.post("/voices", dependencies=[Depends(require_key)])
async def upload_voice(file: UploadFile = File(...)):
    return {"voice_id": save_uploaded_voice(await file.read(), file.filename)}


class TTSRequest(BaseModel):
    text: str
    language: str = "en"
    speed: float = 1.0
    voice_id: Optional[str] = None


@app.post("/tts", dependencies=[Depends(require_key)])
def synthesize(req: TTSRequest):
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="text is empty")
    if state["error"]:
        raise HTTPException(status_code=500, detail=state["error"])
    if state["tts"] is None:
        raise HTTPException(status_code=503, detail="Model is still loading")

    speaker_wav = None
    if req.voice_id:
        speaker_wav = voice_path(req.voice_id)
        if not speaker_wav:
            raise HTTPException(status_code=404, detail="Unknown voice_id - upload the voice sample again")

    if FAKE:
        return Response(content=fake_wav_bytes(text), media_type="audio/wav")

    out_path = os.path.join(VOICE_DIR, f"out_{threading.get_ident()}_{os.getpid()}.wav")
    with synth_lock:
        tts = state["tts"]
        kwargs = dict(text=text, file_path=out_path, language=req.language, split_sentences=True)
        if speaker_wav:
            kwargs["speaker_wav"] = speaker_wav
        else:
            kwargs["speaker"] = state["default_speaker"]
        try:
            tts.tts_to_file(speed=req.speed, **kwargs)
        except TypeError:
            tts.tts_to_file(**kwargs)  # older builds without a speed argument
    with open(out_path, "rb") as fh:
        audio = fh.read()
    os.remove(out_path)
    return Response(content=audio, media_type="audio/wav")


In [ ]:
# Start the TTS server + a public tunnel, then print the URL and API key
import os, re, sys, time, json, secrets, subprocess, urllib.request, urllib.error

os.environ["COQUI_TOS_AGREED"] = "1"     # harmless if this engine doesn't use it
API_KEY = secrets.token_urlsafe(24)      # protects the public URL - don't share it
PORT = 8000

def health():
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health", headers={"X-API-Key": API_KEY})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())

# --- start the server (loads the model in the background) ---
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server_xtts:app", "--host", "127.0.0.1", "--port", str(PORT)],
    env={**os.environ, "RG_API_KEY": API_KEY},
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
print("Loading XTTS-v2 (first run downloads ~2 GB, a few minutes)...")
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open("server.log").read()[-3000:])
        raise SystemExit("Server crashed - see the log above")
    try:
        info = health()
        if info.get("status") == "ok":
            break
    except urllib.error.HTTPError as e:
        if e.code == 500:
            raise SystemExit("Model failed to load: " + e.read().decode()[:1500])
    except Exception:
        pass                                   # server still starting
    if time.time() - t0 > 900:
        raise SystemExit("Timed out waiting for the model to load")
    time.sleep(5)
print("Model ready on:", info.get("gpu") or info.get("device"))

# --- open a public tunnel (Cloudflare quick tunnel: free, no account) ---
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        public_url = m.group(0)
        break
if not public_url:
    print(open("tunnel.log").read()[-2000:])
    raise SystemExit("Could not open a tunnel - see the log above")

print("\n" + "=" * 64)
print(" PASTE THESE INTO RemoGen Studio -> Voice Engine")
print("=" * 64)
print(f" Engine    : XTTS-v2")
print(f" Colab URL : {public_url}")
print(f" API key   : {API_KEY}")
print("=" * 64)
print(" Keep this cell running. Stop it (or close the tab) to shut TTS down.\n")

# --- keep the runtime alive while you work ---
try:
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate(); server.terminate()
    print("Stopped.")

### Troubleshooting
- **"NO GPU"** - Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run all.
- **Install errors about `transformers` / `torch`** - run `!pip -q install -U coqui-tts` and restart the runtime.
- **RemoGen says the tunnel is down** - Colab stopped (free tier disconnects after idle/limits). Re-run the last cell and paste the new URL + key.
- **Voice doesn't sound like your sample** - use 10-30 s of clean speech, one speaker, no music/noise.